# Railway Safety Image Classification

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / 'dataset_clean'
OUTPUT_DIR = PROJECT_DIR / 'outputs_final'
REPORT_PATH = OUTPUT_DIR / 'comparison_report.json'

print('Project:', PROJECT_DIR)
print('Dataset exists:', DATASET_DIR.exists())
print('Report exists:', REPORT_PATH.exists())

In [ ]:
for split in ['train', 'val', 'test']:
    image_count = len(list((DATASET_DIR / 'images' / split).glob('*')))
    label_count = len(list((DATASET_DIR / 'labels' / split).glob('*.txt')))
    print(f'{split:<5} images={image_count:<4} labels={label_count:<4}')

In [ ]:
with REPORT_PATH.open('r', encoding='utf-8') as f:
    report = json.load(f)

print('Dataset:', report['dataset'])
print('Counts:', report['counts'])
print('Class names:', report['class_names'])

In [ ]:
from cv_features import extract_features

example_image = next((DATASET_DIR / 'images' / 'test').glob('*'))
features = extract_features(example_image, image_size=128)

print('Example image:', example_image.name)
print('Feature vector shape:', features.shape)
print('Feature vector length:', len(features))

In [ ]:
print('Feature method:', report['feature_method'])
print('Best by CV danger F1:', report['best_by_cv_danger_f1'])
print('Best by test accuracy:', report['best_by_test_accuracy'])

In [ ]:
model_order = ['linear_svm', 'random_forest', 'hist_gradient_boosting']
display_names = report['model_display_names']

header = f"{'Model':<24} {'CV Acc':>8} {'CV Prec':>8} {'CV Recall':>10} {'CV F1':>8} {'Infer ms/img':>13}"
print(header)
print('-' * len(header))
for key in model_order:
    cv = report['cv_results'][key]
    test = report['test_results'][key]
    print(
        f"{display_names[key]:<24} "
        f"{cv['cv_accuracy_mean']:>8.3f} "
        f"{cv['cv_danger_precision_mean']:>8.3f} "
        f"{cv['cv_danger_recall_mean']:>10.3f} "
        f"{cv['cv_danger_f1_mean']:>8.3f} "
        f"{test['prediction_time_ms_per_image']:>13.3f}"
    )

In [ ]:
for key in model_order:
    matrix = report['test_results'][key]['confusion_matrix']
    print(display_names[key])
    print(matrix)
    print()

In [ ]:
from IPython.display import Image, display

figure_paths = [
    OUTPUT_DIR / 'figures' / 'test_confusion_matrix_linear_svm.png',
    OUTPUT_DIR / 'figures' / 'test_confusion_matrix_random_forest.png',
    OUTPUT_DIR / 'figures' / 'test_confusion_matrix_hist_gradient_boosting.png',
]

for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
for key, model_path in report['exported_models'].items():
    path = PROJECT_DIR / model_path
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f'{display_names[key]:<24} {path} ({size_mb:.3f} MB)')

In [ ]:
print('Best model path:', OUTPUT_DIR / 'best_model.pkl')
print('Final selected model:', display_names[report['best_by_cv_danger_f1']])